# Code pour le suivi des épidémies en RDC - données hebdomadaires DSE

**Auteur** : Marie-Julie Lambert & Alex Kaldjian

**Date** : Janvier 2021

**Projet** : EOC Malaria (Gates)

**Pays** : RDC

## Chargement des librairies et connexion à la base de données PostGres

In [ ]:
options(
    repr.matrix.max.cols = 30,
    repr.plot.width = 12,
    repr.plot.height = 6
)

In [ ]:
install.packages("scales", quiet = TRUE) ## install this extra library
install.packages("zoo", quiet = TRUE)

In [ ]:
library(data.table)
# library(ggplot2)
library(RPostgres) 
library(sf)
library(zoo)
library(stringr)
library(DBI)

In [ ]:
con <- dbConnect(
    RPostgres::Postgres(),
    dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
    host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
    port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
    user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
    password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
)

In [ ]:
# helper functions
corriger_ZS <- function(nom, df) {
    # find IDs of all observations with given zone name
    id <- unique(df[ZS == nom, c(ID)])

    for (i in id) {
        # find names of all zones with that ID
        noms_possibles <- df[ID == i,
            .N,
            by = ZS
        ][order(-N), c(ZS)]

        # chose name with most observations
        nom_choisi <- noms_possibles[1]

        # assign correct name to all relevant observations
        df[ZS == nom & ID == i, ZS := nom_choisi]
    }
    return()
}

extract_year_from_path <- function(path) {
  file_name <- basename(path)
  # Check if filename starts with 4 digits
  if (!grepl("^[0-9]{4}", file_name)) {
    return(NULL)
  }
  
  year <- as.numeric(sub("^([0-9]{4}).*", "\\1", file_name))
  
  # Extra safety check (in case conversion fails)
  if (is.na(year)) {
    return(NULL)
  }  
  return(year)
}

# Function to get the latest week file from a folder
get_latest_week_file <- function(folder_path, pattern = "_Week([0-9]+)\\.csv$") {
  all_files <- list.files(folder_path, full.names = TRUE)
  
  # Filter files matching the week pattern
  week_files <- all_files[str_detect(all_files, pattern)]
  
  # If no files found, return NULL
  if(length(week_files) == 0){
    warning("No files matching the pattern found in folder: ", folder_path)
    return(NULL)
  }
  
  # Extract week numbers
  week_numbers <- str_extract(week_files, pattern) %>% 
    str_extract("[0-9]+") %>% 
    as.integer()
  
  # Return the file with the max week number
  week_files[which.max(week_numbers)]
}

## Chargement des données pour calculs des outbreaks et cleaning des BD

Path vers le nouveau fichier d'input ici:

In [ ]:
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2024/2024_Database_Week05.csv"
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2025/2025_Database_Week18.csv"
# nouvelles_donnees_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2026/2026_Database_Week05.csv"

In [ ]:
# Set paths
base_path <- "/home/hexa/workspace/tdb-suivi-epidemio/data/"
raw_data_path <- file.path(base_path, "raw/")
metadata_path <- file.path(base_path, "metadata/")
output_path <- file.path(base_path, "outputs/")
historic_path <- file.path(base_path, "raw", "historic")

### Load current year data 

In [ ]:
# Retrieve year from input filepath
current_year <- extract_year_from_path(nouvelles_donnees_path)
if (is.null(current_year)) {
    stop("Impossible de trouver l’année de référence dans le fichier. Le processus est arrêté.")
}
current_year

In [ ]:
# Load file (current year - week)
print(glue::glue("Chargement des données pour l’année {current_year}: {nouvelles_donnees_path} "))
dse_data <- read.csv(nouvelles_donnees_path, sep = ";")  

# Handle comma separated file format (usually saved after manual checking)
if (length(colnames(dse_data)) == 1) { dse_data <- read.csv(nouvelles_donnees_path, sep = ",") }
print(colnames(dse_data))

setDT(dse_data)
dse_data[, C1259MOIS := as.integer(C1259MOIS)]
dse_data[, year := current_year]  # Update YEAR!

dse_data[, TOTALDECES := as.integer(TOTALDECES)]
dse_data[is.na(TOTALCAS),
    TOTALCAS := rowSums(.SD, na.rm = T),
    .SDcols = c("C011MOIS", "C1259MOIS", "C515ANS", "CP15ANS")
]
dse_data[is.na(TOTALDECES),
    TOTALDECES := rowSums(.SD, na.rm = T),
    .SDcols = c("D011MOIS", "D1259MOIS", "D515ANS", "DP15ANS")
]

### Load previous years data 

In [ ]:
# Load DSE folder data
all_dirs <- list.dirs(raw_data_path, full.names = TRUE, recursive = FALSE)
dse_folders <- all_dirs[grepl("DSE_data_", basename(all_dirs))]
dse_folders <- dse_folders[!grepl(as.character(current_year), basename(dse_folders))] # except current year folder

# Load last available file from each folder
dse_prev_years <- data.table()
for (dse_folder in dse_folders) {

    dse_filename <- get_latest_week_file(dse_folder)
    file_year <- extract_year_from_path(dse_filename)
    print(glue::glue("Chargement des données pour l’ancienne année {file_year}: {dse_filename}"))
    
    dse_data_year <-  fread(dse_filename, sep = ";", encoding = "Latin-1", quote = "")  # or "UTF-8"
    # Handle comma separated file format (usually saved after manual checking)
    if (length(colnames(dse_data_year)) < 3) {         
        dse_data_year <-  fread(dse_filename, sep = ",", encoding = "Latin-1", quote = "")  # or "UTF-8" 
    }
    
    setDT(dse_data_year)
    dse_data_year[, year := file_year]  # Update YEAR!
    
    if (file_year != 2021) {
        dse_data_year[, C1259MOIS := as.integer(C1259MOIS)]        
        dse_data_year[, TOTALDECES := as.integer(TOTALDECES)]
        dse_data_year[is.na(TOTALCAS),
            TOTALCAS := rowSums(.SD, na.rm = T),
            .SDcols = c("C011MOIS", "C1259MOIS", "C515ANS", "CP15ANS")
        ]
        dse_data_year[is.na(TOTALDECES),
            TOTALDECES := rowSums(.SD, na.rm = T),
            .SDcols = c("D011MOIS", "D1259MOIS", "D515ANS", "DP15ANS")
        ]
    }
    
    dse_prev_years <- rbind(dse_prev_years, dse_data_year, fill = TRUE)
}

### Load historic data 

In [ ]:
# Load DSE historic data
dse_hits_files <- list.files(historic_path, full.names = TRUE)

dse_hist <- data.table()
for (dse_hist_file in dse_hits_files) {
    file_year <- extract_year_from_path(dse_hist_file)
    print(glue::glue("Chargement des données historiques pour l’année {file_year}: {dse_hist_file}"))
    dse_hist_year <- fread(dse_hist_file, sep = ";")    
    dse_hist_year[, year := file_year]

    if (file_year == 2020) {
       dse_hist_year[DEBUTSEM == "28/12/2021", DEBUTSEM := "28/12/2020"] 
    }
    if (file_year == 2018) {
        dse_hist_year[NUMSEM == 0, NUMSEM := 1]
    }
    
    dse_hist <- rbind(dse_hist, dse_hist_year, fill = TRUE)
}

In [ ]:
# concat data
combo <- rbind(dse_hist, dse_prev_years, dse_data, fill = TRUE)

In [ ]:
dim(combo)
head(combo, 3)

In [ ]:
# Check all years are listed here
unique(combo$year)

# Nettoyage des données

In [ ]:
# all sorts of different province names
sort(unique(combo$PROV))

In [ ]:
# fix province names + drop empties
combo[, PROV := gsub("-", " ", PROV)]
combo[PROV == "LULABA", PROV := "LUALABA"]
combo <- combo[PROV != ""]

# fix disease mispellings
combo[, MALADIE := toupper(MALADIE)]
combo[MALADIE == "MONKEYPOX", MALADIE := "MONKEY POX"]
combo[MALADIE == "DECES MATERNEL", MALADIE := "DECES MATERNELS"]
combo <- combo[MALADIE != ""]

# drop MAPI
combo <- combo[MALADIE != "MAPI"]

# drop NA weeks
combo <- combo[!is.na(NUMSEM)]
combo <- combo[NUMSEM != 0]
combo <- combo[NUMSEM < 53]

# reduce to necessary columns
combo <- combo[, .(
    NUM, PROV, ZS, NUMSEM, MALADIE, C328TNN, DTNN, C011MOIS, D011MOIS,
    C1259MOIS, D1259MOIS, C5ANSP, D5ANSP, C515ANS, D515ANS, CP15ANS, DP15ANS,
    TOTALCAS, TOTALDECES, year
)]

In [ ]:
sort(unique(combo$PROV))
tot_diff_prov <- length(sort(unique(combo$PROV)))
print(tot_diff_prov)
if (tot_diff_prov != 26) {
    stop("Number of unique province names in the dataset is not 26: {}. Process halted.")
}

In [ ]:
combo[, NUM := iconv(NUM, from = "", to = "UTF-8", sub = "byte")] # Force R to convert all values in NUM to UTF-8 and replaces invalid characters
combo[, ID := gsub("[[:digit:]]", "", NUM)]

noms_a_corriger <- combo[, .N, by = .(ZS, PROV)][N < 250, c(ZS)]

invisible(lapply(noms_a_corriger, corriger_ZS, combo))

In [ ]:
combo[, .N, by = .(PROV, ZS)][N < 200]

In [ ]:
# fix the last few, then drop the rest
combo <- merge(combo,
    combo[, .N, by = .(PROV, ZS)],
    all.x = T
)

combo[N < 200, ZS := gsub(" ", "", ZS)]

combo <- combo[N > 100]

In [ ]:
all_names <- combo[, .N, by = .(PROV, ZS)][order(N)]

In [ ]:
# write out to match with love machine
fwrite(unique(combo[, .(zs_name = ZS, province = PROV)]),
    paste0(metadata_path, "ZS_names_all.csv")
)

## Import zone de santé dictionary from love machine

In [ ]:
zs_dict <- fread(
    paste0(metadata_path, "love_machine_zone_dict.csv"),
    header = TRUE
)

In [ ]:
zs_dict <- zs_dict[, .(
    ZS = zone_DSE, PROV = province_DSE,
    Nom = zone_ref, PROVINCE = province_ref,
    zone_id_DHIS2, province_id_DHIS2
)]

In [ ]:
test_allx <- merge(all_names, zs_dict, all.x = T, by = c("ZS", "PROV"))

Have to fix a few manually...

In [ ]:
test_allx[is.na(Nom)]

In [ ]:
manually_corrected <- fread(
    paste0(metadata_path, "manually_corrected_full.csv"),
    header = TRUE
)

manually_corrected <- manually_corrected[, .(
    ZS = zone_DSE, PROV = province_DSE,
    Nom = zone_ref, PROVINCE = province_ref,
    zone_id_DHIS2, province_id_DHIS2
)]

In [ ]:
zs_dict <- rbind(zs_dict[!(ZS %in% manually_corrected$ZS)], manually_corrected)

In [ ]:
test_allx <- merge(all_names, zs_dict, all.x = T, by = c("ZS", "PROV"))

In [ ]:
test_allx[is.na(Nom)]

### Disease-weeks reported by zone de santé

In [ ]:
combo <- merge(combo, zs_dict, by = c("ZS", "PROV"), all.x = T)

In [ ]:
# ggplot(aes(N), data = combo[, .N, by = .(Nom, PROVINCE)]) +
    # geom_histogram(bins = 40)

## Imputing 0s for non-reported diseases (from reported zone-weeks)

In [ ]:
# merge all data on rectangularized dataset

# combo[, merge_ID := paste0(Nom, "_", PROVINCE)]

ge <- expand.grid(
    zone_id_DHIS2 = unique(combo$zone_id_DHIS2), year = unique(combo$year),
    NUMSEM = unique(combo$NUMSEM), MALADIE = unique(combo$MALADIE)
)
setDT(ge)

combo <- merge(ge, combo, by = c("zone_id_DHIS2", "year", "NUMSEM", "MALADIE"), all.x = T)

In [ ]:
# compute number of non-NA observations by week
zone_weeks <- combo[, lapply(.SD, function(x) sum(!is.na(x))),
    by = .(zone_id_DHIS2, NUMSEM, year), .SDcols = "ZS"
]
zone_weeks[, reported := as.logical(ZS)]

In [ ]:
combo <- merge(combo, zone_weeks[, .(zone_id_DHIS2, NUMSEM, year, reported)],
    by = c("zone_id_DHIS2", "year", "NUMSEM"), all.x = TRUE
)

combo <- combo[reported == TRUE]

combo[, imputed := FALSE][is.na(Nom) & reported, imputed := TRUE]

In [ ]:
combo[reported == TRUE, TOTALCAS := rowSums(.SD, na.rm = T),
    .SDcols = c("C328TNN", "C011MOIS", "C1259MOIS", "C515ANS", "C5ANSP", "CP15ANS")
]
combo[reported == TRUE, TOTALDECES := rowSums(.SD, na.rm = T),
    .SDcols = c("DTNN", "D011MOIS", "D1259MOIS", "D515ANS", "D5ANSP", "DP15ANS")
]

combo[, Nom := na.locf0(Nom, fromLast = TRUE), by = zone_id_DHIS2][, Nom := na.locf0(Nom), by = zone_id_DHIS2]
combo[, PROVINCE := na.locf0(PROVINCE,
    fromLast = TRUE
),
by = zone_id_DHIS2
][, PROVINCE := na.locf0(PROVINCE), by = zone_id_DHIS2]

combo[imputed == TRUE & MALADIE == "TNN", 
      C328TNN := nafill(C011MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE & MALADIE == "TNN", 
      DTNN := nafill(D011MOIS, fill = 0, nan = NA)]

combo[imputed == TRUE, C011MOIS := nafill(C011MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, D011MOIS := nafill(D011MOIS, fill = 0, nan = NA)]

combo[imputed == TRUE, C1259MOIS := nafill(C1259MOIS, fill = 0, nan = NA)]
combo[imputed == TRUE, D1259MOIS := nafill(D1259MOIS, fill = 0, nan = NA)]

combo[imputed == TRUE, C515ANS := nafill(C515ANS, fill = 0, nan = NA)]
combo[imputed == TRUE, D515ANS := nafill(D515ANS, fill = 0, nan = NA)]

combo[imputed == TRUE, C5ANSP := nafill(C5ANSP, fill = 0, nan = NA)]
combo[imputed == TRUE, D5ANSP := nafill(D5ANSP, fill = 0, nan = NA)]

combo[imputed == TRUE, CP15ANS := nafill(CP15ANS, fill = 0, nan = NA)]
combo[imputed == TRUE, DP15ANS := nafill(DP15ANS, fill = 0, nan = NA)]

combo[imputed == TRUE, TOTALCAS := nafill(TOTALCAS, fill = 0, nan = NA)]
combo[imputed == TRUE, TOTALDECES := nafill(TOTALDECES, fill = 0, nan = NA)]

## Outbreak detection

In [ ]:
alert_outbreak <- function(alerte_type, cholera_type,
                           cas, lag_cas, deces) {

    # 2: alert
    # 1: no alert but cases present
    # 0: no cases

    if (alerte_type == "doublement") {
        alert <- (cas >= (2 * lag_cas)) & (cas != 0)
    } else if (alerte_type == "cholera") {
        if (cholera_type %in% c("C", "D")) {
            alert <- as.logical(cas)
        } else {
            alert <- (cas >= (2 * lag_cas)) & (cas != 0)
        }
    } else if (alerte_type == "cas_unique") {
        alert <- as.logical(cas)
    } else if (alerte_type == "doublement_letalite") {
        double <- (cas >= (2 * lag_cas)) & (cas != 0)
        letal_2pct <- (deces / cas) > 0.02
        alert <- double & letal_2pct
    } else if (alerte_type == "cas_plus_deces") {
        alert <- as.logical(cas) & as.logical(deces)
    } else {
        stop("Invalid alert type")
    }

    if (is.na(cas)) {
        return(NA)
    } else if ((is.na(alert) | !alert) & cas > 0) {
        return(1)
    } else if (alert) {
        return(2)
    } else {
        return(0)
    }
}

In [ ]:
alert_df <- combo[order(year, NUMSEM)][, .(
    TOTALCAS = sum(TOTALCAS),
    TOTALDECES = sum(TOTALDECES)
),
by = .(zone_id_DHIS2, MALADIE, year, NUMSEM)
]

In [ ]:
alert_df <- alert_df[
    order(year, NUMSEM)
][, .(year, NUMSEM,
    LAGCAS = shift(TOTALCAS),
    TOTALCAS, TOTALDECES
),
by = .(zone_id_DHIS2, MALADIE)
]

In [ ]:
config <- fread(
    paste0(
        metadata_path,
        "alerte_config.csv"
    )
)
cholera_zones <- fread(
    paste0(
        metadata_path,
        "cholera_zones.csv"
    )
)

names(cholera_zones)[names(cholera_zones) == "Type"] <- "cholera_type"

In [ ]:
alert_df <- merge(alert_df, config)

alert_df <- merge(alert_df, cholera_zones,
    all.x = TRUE,
    by = "zone_id_DHIS2"
)
alert_df[is.na(cholera_type), cholera_type := "D"]

In [ ]:
head(alert_df)

In [ ]:
alert_df[, ALERTE := mapply(
    alert_outbreak, alert_type, cholera_type,
    TOTALCAS, LAGCAS, TOTALDECES
)]

In [ ]:
alert_df <- alert_df[, .(zone_id_DHIS2, MALADIE, year, NUMSEM, ALERTE)]

# Organize data for export

In [ ]:
out <- combo[, .(
    zone_id_DHIS2, year, NUMSEM, MALADIE,
    C328TNN, DTNN, 
    C011MOIS, D011MOIS,
    C1259MOIS, D1259MOIS, 
    C5ANSP, D5ANSP, 
    C515ANS, D515ANS, 
    CP15ANS, DP15ANS)]

In [ ]:
# this may be generating NAs, not sure what else to do though
out <- out[, .(
    C328TNN = sum(C328TNN), 
    DTNN = sum(DTNN),
    C011MOIS = sum(C011MOIS),
    D011MOIS = sum(D011MOIS),
    C1259MOIS = sum(C1259MOIS),
    D1259MOIS = sum(D1259MOIS),
    C5ANSP = sum(C5ANSP),
    D5ANSP = sum(D5ANSP),
    C515ANS = sum(C515ANS), 
    D515ANS = sum(D515ANS),
    CP15ANS = sum(CP15ANS), 
    DP15ANS = sum(DP15ANS)
),
by = .(zone_id_DHIS2, year, NUMSEM, MALADIE)
]

In [ ]:
cases <- melt(out,
    id.vars = c("zone_id_DHIS2", "year", "MALADIE", "NUMSEM"),
    measure.vars = c("C328TNN", "C011MOIS", "C1259MOIS", "C515ANS", "C5ANSP", "CP15ANS")
)

In [ ]:
names(cases)[5:6] <- c("Cas_Age", "Cas")

In [ ]:
deaths <- melt(out,
    id.vars = c("zone_id_DHIS2", "year", "MALADIE", "NUMSEM"),
    measure.vars = c("DTNN", "D011MOIS", "D1259MOIS", "D515ANS", "D5ANSP", "DP15ANS")
)
names(deaths)[5:6] <- c("Deces_Age", "Deces")

In [ ]:
out <- cbind(cases, deaths[, 5:6])

In [ ]:
# correction for changes in data collection
new.maladies.2022 <- c(
    "PALUDISME CONF", "PALUDISME SUSP", "MAPI LEGERES",
    "PNEUMONIE", "DIPHTERIE", "TETANOS MATERNE",
    "MAPI GRAVES"
)

out <- out[(year > 2019 & MALADIE == "COVID-19") |
    (year > 2021 & !(MALADIE %in% c("PALUDISME", "PALUDISME TDR +"))) |
    (year < 2022 & !(MALADIE %in% new.maladies.2022))]

out <- out[(year < 2021 & !(Cas_Age %in% c("C328TNN", "C515ANS", "CP15ANS"))) |
    (year > 2020 & !(Cas_Age == c("C5ANSP")))]

In [ ]:
out <- merge(out,
    alert_df,
    by = c("zone_id_DHIS2", "year", "MALADIE", "NUMSEM"),
    all.x = T
)

In [ ]:
zones.DHIS <- read_sf(
    paste0(
        base_path,
        'geodata/org_units-2021-10-21-14-52_level_Zone_de_Sante.gpkg'
    )
)

In [ ]:
setDT(zones.DHIS)
zones.DHIS <- zones.DHIS[, .(ZONE = name,
                             zone_id_DHIS2 = ref, 
                             PROVINCE = parent,
                             province_id_DHIS2 = parent_ref)]
zones.DHIS[, PROVINCE := gsub(' \\(Province\\)', '', PROVINCE)]

In [ ]:
out <- merge(out, zones.DHIS, by = 'zone_id_DHIS2')

In [ ]:
out[, Date := as.Date(paste(year, NUMSEM, 1, sep = "-"), "%Y-%U-%u")]

In [ ]:
out <- out[Date < Sys.Date()]

In [ ]:
nrow(out)

In [ ]:
head(out)

In [ ]:
tail(out)

## Output data to Tableau DB

In [ ]:
# fwrite(
#     out,
#     paste0(
#         output_path,
#         "latest.csv"
#     )
# )

In [ ]:
# dbWriteTable(con.ocha, "DSE_MAPEPI", out, overwrite = TRUE)
# dbWriteTable(con, "DSE_Surv_Epi_Complete", out, overwrite = TRUE)
# print("Upload complete")

In [ ]:
# BATCH UPLOAD

# Define batch size
batch_size <- 500000  # adjust depending on memory/performance

# Total number of rows
n <- nrow(out)

# First write: create the table structure with the first chunk
# dbWriteTable(con, "DSE_Surv_Epi_Complete", out[1:batch_size, ], overwrite = TRUE)

# # Append remaining batches
# for (i in seq(batch_size + 1, n, by = batch_size)) {
#     end <- min(i + batch_size - 1, n)
#     print(paste0(i, " - ", end))
#     dbWriteTable(con, "DSE_Surv_Epi_Complete", out[i:end, ], append = TRUE)
# }
print("Upload complete")